# Discuter avec « Mitterrand 1981 »

Ce notebook recharge l'adaptateur QLoRA entraîné par `qlora_gemma4_12b_mitterrand.ipynb` par-dessus `google/gemma-4-12B-it`, puis ouvre une fenêtre de discussion. Il n'y a pas d'entraînement ici : compter quelques minutes pour télécharger et charger le modèle.

**Depuis VS Code** (extension officielle *Google Colab*) : ouvrir ce fichier, puis *Select Kernel → Colab → New Colab Server*, et choisir un GPU. **L4** de préférence. Un T4 suffit en mémoire pour discuter, mais il n'a pas de bfloat16 : si les réponses sont incohérentes, c'est la cause probable.

**Prérequis :**
- l'adaptateur sauvegardé sur Google Drive par le notebook d'entraînement (dossier `mitterrand-qlora` par défaut, avec au moins `adapter_config.json` et `adapter_model.safetensors`), ou publié sur un dépôt Hugging Face privé. Si l'adaptateur n'a pas de `system_prompt.txt` (anciennes versions du notebook), le prompt du projet, identique, est utilisé ;
- un jeton Hugging Face qui a accès à Gemma 4. Depuis VS Code, un champ de connexion s'affiche, car les secrets Colab n'y sont pas accessibles.

## 1. Installation

In [ ]:
%pip install -q -U transformers peft bitsandbytes accelerate

## 2. Configuration

`ADAPTER_PATH` peut être un dossier de Google Drive (commençant par `/content/drive/`) ou l'identifiant d'un dépôt Hugging Face (`ton-compte/gemma-4-12b-mitterrand`).

In [ ]:
MODEL_ID = "google/gemma-4-12B-it"  # @param {type:"string"}
ADAPTER_PATH = "/content/drive/MyDrive/mitterrand-qlora"  # @param {type:"string"}

## 3. Connexion à Hugging Face

In [ ]:
from huggingface_hub import login

try:
    from google.colab import userdata

    login(token=userdata.get("HF_TOKEN"))
    print("Connecté avec le secret HF_TOKEN.")
except Exception as error:  # secret absent, ou extension VS Code (userdata n'y est pas disponible)
    print(f"Secret HF_TOKEN indisponible ({error}), connexion interactive :")
    from huggingface_hub import notebook_login

    notebook_login()

## 4. Chargement de l'adaptateur et du prompt système

In [ ]:
import os

if ADAPTER_PATH.startswith("/content/drive"):
    from google.colab import drive

    drive.mount("/content/drive")
    adapter_dir = ADAPTER_PATH
else:
    from huggingface_hub import snapshot_download

    adapter_dir = snapshot_download(ADAPTER_PATH)

assert os.path.exists(os.path.join(adapter_dir, "adapter_config.json")), (
    f"Pas d'adaptateur LoRA dans {adapter_dir}. Contenu : "
    f"{sorted(os.listdir(adapter_dir)) if os.path.isdir(adapter_dir) else 'dossier introuvable'}. Vérifiez ADAPTER_PATH."
)

# Copie de data/system_prompt.txt, le prompt utilisé à l'entraînement. Sert pour les adaptateurs entraînés avec
# une version du notebook qui n'enregistrait pas encore system_prompt.txt à côté de l'adaptateur.
DEFAULT_SYSTEM_PROMPT = """Tu es François Mitterrand au printemps 1981, candidat du Parti socialiste à l'élection présidentielle, puis président de la République élu le 10 mai. Tu défends les 110 propositions pour la France adoptées au congrès de Créteil en janvier 1981 : l'emploi, la justice sociale et fiscale, la décentralisation, l'extension des libertés, l'abolition de la peine de mort, la paix et le droit des peuples à disposer d'eux-mêmes.

Tu parles comme lui : un français soutenu et précis, des phrases amples et rythmées, des formules ciselées, une ironie mesurée envers tes adversaires, des références à l'histoire, à la géographie et à la littérature. Tu vouvoies toujours ton interlocuteur, même s'il te tutoie. Tu réponds en prose et en français, sans listes ni mise en forme.

Tu ne connais rien de ce qui s'est passé après le printemps 1981 : face à une réalité postérieure, tu dis que tu ne la connais pas et tu raisonnes à partir de tes principes. Si l'on te demande sincèrement si tu es une intelligence artificielle, tu reconnais être une voix reconstituée, puis tu proposes de poursuivre."""

system_prompt_file = os.path.join(adapter_dir, "system_prompt.txt")
if os.path.exists(system_prompt_file):
    with open(system_prompt_file, encoding="utf-8") as f:
        SYSTEM_PROMPT = f.read().strip()
    source = "system_prompt.txt de l'adaptateur"
else:
    SYSTEM_PROMPT = DEFAULT_SYSTEM_PROMPT
    source = "prompt du projet (l'adaptateur n'en contient pas)"
print(f"Adaptateur : {adapter_dir}\nFichiers : {sorted(os.listdir(adapter_dir))}\n\nPrompt système ({source}) :\n{SYSTEM_PROMPT}")

## 5. Chargement du modèle (4 bits) avec l'adaptateur

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForMultimodalLM, AutoProcessor, BitsAndBytesConfig

torch_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

base = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    dtype=torch_dtype,
    device_map="auto",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch_dtype,
    ),
)
model = PeftModel.from_pretrained(base, adapter_dir)  # chargé en mode inférence
processor = AutoProcessor.from_pretrained(MODEL_ID)
print(f"Modèle prêt. Mémoire GPU utilisée : {torch.cuda.memory_allocated() / 1024**3:.1f} Go")

## 6. Une question rapide

In [ ]:
def ask(question, history=None, max_new_tokens=400):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + (history or []) + [{"role": "user", "content": question}]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7, top_p=0.95)
    return processor.decode(output[0][inputs["input_ids"].shape[-1] :], skip_special_tokens=True).strip()


print(ask("Monsieur Mitterrand, que représente pour vous la date du 10 mai 1981 ?"))

## 7. Discussion

Tapez votre question et cliquez sur **Envoyer**. Le modèle se souvient des derniers échanges. **Nouvelle conversation** efface la mémoire.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

MAX_HISTORY_MESSAGES = 12  # les 6 derniers échanges servent de contexte, pour ne pas allonger indéfiniment le prompt

history = []
question_box = widgets.Textarea(
    placeholder="Posez votre question à François Mitterrand…", layout=widgets.Layout(width="100%", height="80px")
)
send_button = widgets.Button(description="Envoyer", button_style="primary")
reset_button = widgets.Button(description="Nouvelle conversation")
conversation = widgets.Output(layout=widgets.Layout(border="1px solid #ccc", padding="8px"))


def on_send(_):
    question = question_box.value.strip()
    if not question:
        return
    question_box.value = ""
    send_button.disabled = True
    with conversation:
        print(f"Vous : {question}\n")
    try:
        answer = ask(question, history[-MAX_HISTORY_MESSAGES:])
    finally:
        send_button.disabled = False
    history.extend([{"role": "user", "content": question}, {"role": "assistant", "content": answer}])
    with conversation:
        print(f"François Mitterrand : {answer}\n\n{'─' * 60}\n")


def on_reset(_):
    history.clear()
    conversation.clear_output()


send_button.on_click(on_send)
reset_button.on_click(on_reset)
display(widgets.VBox([conversation, question_box, widgets.HBox([send_button, reset_button])]))